# EXP-040 · E3-LoRA (Qwen3-VL-Embedding-2B adapted from a ridge-initialized interaction head)

Attach **two private datasets**: the existing EXP-037 input (`exp037_inputs.zip`) and the EXP-040 overlay
(`exp040_inputs.zip`). Select **GPU T4 ×2** and **Internet on**. Each GPU runs its own child process with the
whole model on that GPU (never split): outer folds run two at a time, and the full refit runs afterwards on GPU 0.

1. **Smoke first:** keep `SMOKE_ONLY = True`, set `AUTHORIZE_RUN = True`, **Run All**. It validates the inputs, the
   key audit, the step-0 equality and the vision-cache check, then trains fold 1 on GPU 0 and fold 2 on GPU 1 at
   the same time for 30 optimizer steps each, runs their inner passes, and writes `exp040_smoke.json` (per-GPU
   throughput, peak VRAM, placement, finite losses, two-pass determinism, the parallel full-run forecast).
   No submission is produced.
2. **Full run:** set `SMOKE_ONLY = False` and **Run All** once. Stages: Setup → Validate → Benchmark (both GPUs) →
   P2 LoRA folds (two GPU queues, sealed per fold, resumable) → P2 evaluation → Full refit (GPU 0) and Development →
   Submission validation → Export → Release. If the forecast exceeds the session, the run stops before training
   and exports a NO_GO report.
   C2 is recomputed from the sealed EXP-038 features and is the gate reference. Development is reporting only.

Keep everything private. Download from **Output**: `exp040_smoke.json`, or `exp040_results.zip`,
`exp040_best_submission.zip` and their `.sha256` files (or `exp040_diagnostic.zip`; `exp040_progress.zip` to
resume). Every ZIP is `UNCONFIRMED_QWEN_ST3`. Never upload from here.


In [ ]:
AUTHORIZE_RUN = False
SMOKE_ONLY = True   # Run the smoke first; set False for the full run
RESUME_INPUT = ""   # Optional absolute path to a previous exp040_progress.zip / exp040_results.zip or its extracted root
E3_SNAPSHOT = ""    # Optional local snapshot of Qwen/Qwen3-VL-Embedding-2B at the pinned revision
SESSION_HOURS = 10.5


In [ ]:
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import shutil
import subprocess
import sys
import time
from zipfile import ZipFile

ARCHIVE_SUFFIXES = (".zip", ".tar", ".gz", ".tgz", ".7z", ".bz2", ".xz", ".whl")
PATCH2 = ("This overlay predates the EXP-038 patch-2 packaging, which stores nested archives as .b64 text so that "
          "Kaggle cannot expand them. Rebuild it with scripts/package_exp038.py from patch 2 or later and reattach it.")

def sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def verify_root(root, manifest_name, manifest_sha):
    # Checks the stored (mounted) members exactly as packaged; .b64 members are decoded later by the launcher.
    manifest_path = root / manifest_name
    if sha256(manifest_path) != manifest_sha:
        raise ValueError(f"Manifest checksum differs at {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    for name, expected in manifest["files"].items():
        relative = PurePosixPath(name)
        target = root / relative
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError(f"Unsafe manifest member {name}")
        if target.is_dir() and name.lower().endswith(ARCHIVE_SUFFIXES):
            raise ValueError(f"{target} is a directory, not the archive listed in the manifest: Kaggle expanded "
                             f"a nested archive. {PATCH2}")
        if not target.is_file() or sha256(target) != expected:
            raise ValueError(f"Missing or corrupt file {target}. Reattach the verified private dataset.")
    return manifest

def locate(mount, work, manifest_name, manifest_sha, archive_name, archive_sha, prefix):
    # Layouts: (a) expanded by Kaggle, (b) the ZIP mounted intact, (c) any deeper recursive expansion.
    manifests = sorted(p for p in mount.rglob(manifest_name) if p.is_file() and (p.parent / "runtime").is_dir())
    expanded = [p.parent for p in manifests if sha256(p) == manifest_sha]
    others = [p.parent for p in manifests if p.parent not in expanded]
    archives = [p for p in sorted(mount.rglob(archive_name)) if p.is_file()]
    intact = [p for p in archives if sha256(p) == archive_sha]
    if len(expanded) > 1 or len(intact) > 1:
        raise ValueError(f"Several copies of {archive_name} are attached ({expanded + intact}). Attach exactly one.")

    def from_archive(path):
        target = work / manifest_sha[:16]
        shutil.rmtree(target, ignore_errors=True)
        target.mkdir(parents=True)
        with ZipFile(path) as archive:
            archive.extractall(target)
        return target / prefix

    if expanded:
        try:
            return expanded[0], verify_root(expanded[0], manifest_name, manifest_sha)
        except ValueError as error:
            if not intact:
                raise
            try:
                verify_root(from_archive(intact[0]), manifest_name, manifest_sha)
                state = "verified"
            except ValueError as second:
                state = f"also failed: {second}"
            raise ValueError(f"The expanded copy at {expanded[0]} failed verification: {error} The intact archive "
                             f"{intact[0]} {state}. Nothing was selected automatically; attach only one copy.")
    if intact:
        root = from_archive(intact[0])
        return root, verify_root(root, manifest_name, manifest_sha)
    raise FileNotFoundError(f"Attach exactly one private dataset with the pinned {archive_name}. Found "
                            f"{len(others)} expanded copies and {len(archives)} archives with other checksums.")

def stream(command, write=None, env=None):
    # Read the child through a pipe and print each line once; the child never writes to the kernel's stdout.
    write = write or (lambda line: print(line, end="", flush=True))
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", env=env) as child:
        for line in child.stdout:
            write(line)
        code = child.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)


In [ ]:
WORKER_SOURCE = "53d2c527c5a2ed13dcc3e072a3f5560b2c3056b5"
BASE_MANIFEST_SHA256 = "026720f406c02236aeaf5bfdc806853816323b66267f81f0e3e2dbbfe1ce7e0e"
BASE_ARCHIVE_SHA256 = "eaf16b9301cb131fbdf61a1b86fb191d3617b2141e39398f0c3b12bcf57ffce2"
OVERLAY_MANIFEST_SHA256 = "c5d2834019fae58444d9c67d9457249589737dc3ad52126ab513ee3ae84a934f"
OVERLAY_ARCHIVE_SHA256 = "4f316e9da2bd5bec35c7f63f5c8446bda78456584b91d5a1110715f01ec3a497"

STARTED = time.time()
MOUNT, OUTPUT = Path("/kaggle/input"), Path("/kaggle/working")
if not AUTHORIZE_RUN:
    raise RuntimeError("Attach both private inputs, enable GPU and Internet, set AUTHORIZE_RUN = True, then Run All.")
if not MOUNT.is_dir() or not OUTPUT.is_dir():
    raise RuntimeError("Open this notebook in Kaggle with both private inputs attached.")
probe = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True) if Path("/dev/nvidia0").exists() else None
if probe is None or probe.returncode != 0 or "GPU 0" not in probe.stdout:
    raise RuntimeError("Select a CUDA GPU in Kaggle Settings, then Run All again.")
print(probe.stdout.strip())
if "GPU 1" not in probe.stdout:
    print("WARNING: one GPU is visible. The run falls back to a single fold queue; select GPU T4 x2 for two.")
base_root, base_manifest = locate(MOUNT, Path("/kaggle/temp/exp040_base"), "exp037_manifest.json",
                                  BASE_MANIFEST_SHA256, "exp037_inputs.zip", BASE_ARCHIVE_SHA256, "exp037_inputs")
overlay_root, overlay_manifest = locate(MOUNT, Path("/kaggle/temp/exp040_overlay_zip"), "exp040_manifest.json",
                                        OVERLAY_MANIFEST_SHA256, "exp040_inputs.zip", OVERLAY_ARCHIVE_SHA256,
                                        "exp040_inputs")
if overlay_manifest["source"] != WORKER_SOURCE or overlay_manifest["base"]["manifest_sha256"] != BASE_MANIFEST_SHA256:
    raise ValueError("The overlay was prepared for a different worker or base dataset")
print("Verified", len(base_manifest["files"]), "base files and", len(overlay_manifest["files"]), "overlay files;",
      len(overlay_manifest.get("encoded", {})), "archives are stored as .b64 and restored by the launcher")


In [ ]:
launcher = overlay_root / "runtime/scripts/exp040_kaggle.py"
command = [sys.executable, "-u", str(launcher), "--base", str(base_root), "--overlay", str(overlay_root),
           "--base-sha", BASE_MANIFEST_SHA256, "--overlay-sha", OVERLAY_MANIFEST_SHA256,
           "--started", str(STARTED), "--hours", str(SESSION_HOURS)]
for option, value in (("resume", RESUME_INPUT), ("e3-snapshot", E3_SNAPSHOT)):
    if value:
        path = Path(value)
        if not path.is_absolute() or not path.exists():
            raise FileNotFoundError(f"Set {option} to its actual absolute Kaggle path: {path}")
        command.extend((f"--{option}", str(path)))
if SMOKE_ONLY:
    command.append("--smoke")
stream(command, env=os.environ | {"PYTHONDONTWRITEBYTECODE": "1"})


In [ ]:
run_dir = OUTPUT / "exp040_run"
for name in ("status.json", "budget_plan.json", "decision.json", "selection.json"):
    path = run_dir / name
    if path.is_file():
        print(name, json.dumps(json.loads(path.read_text()), indent=1)[:4000])
smoke = OUTPUT / "exp040_smoke.json"
if smoke.is_file():
    result = json.loads(smoke.read_text())
    print("SMOKE_ONLY receipt:", json.dumps({k: result[k] for k in ("validate", "vision_cache", "per_gpu", "forecast")},
                                            indent=1)[:9000])
for name in ("exp040_SUCCESS.json", "exp040_results.zip", "exp040_best_submission.zip", "exp040_progress.zip",
             "exp040_diagnostic.zip"):
    path = OUTPUT / name
    if path.is_file():
        receipt = OUTPUT / f"{name}.sha256"
        print(path, receipt.read_text().strip() if receipt.is_file() else "")
